# AIMLZC416 — Mathematical Foundations for Machine Learning
## Assignment I — Solutions
**M.Tech in Artificial Intelligence & Machine Learning, WILP, BITS Pilani — I Semester 2026-27**

---
* **Name:** `Prahlad Vinayak Avadhani`
* **BITS ID:** `2026AJ05253`
* **Course:** `AIMLZC416 — Mathematical Foundations for Machine Learning`
* **Assignment:** `Assignment 1`
---

### Core Implementation Guidelines
To demonstrate a first-principles mathematical understanding of the algorithms, the following constraints are adhered to throughout this notebook:
1. **From-Scratch Implementations:** All core matrix operations, Gaussian elimination (REF/RREF), particular/nullspace/general solutions, covariance estimation, and the Power Method with deflation are implemented manually using pure Python loops and basic arithmetic. No built-in linear algebra solvers (such as `numpy.linalg`, `scipy`, or `sympy`) are used for these tasks.
2. **Numerical Format:** All hardcoded entries and random coefficients are specified in the required decimal format (`n.dddddddd` with exactly 8 decimal places).
3. **Robust Edge-Case Handling:** Scenarios like zero-pivots or division-by-zero are explicitly detected and handled.
4. **Standard Library Imports:** Standard imports are avoided in Q1 to ensure a purely independent implementation, and utilized appropriately in Q2 for normal feature generation and verification.


## Q1) Finding Solutions of Linear Systems

### 1.1 REF and RREF of $[A \mid b]$

**Methodology & Algorithm:**

1. **Forward Elimination (Row Echelon Form - REF):**
   * **Augmentation:** Construct $[A \mid b]$ of size $m \times (n+1)$.
   * **Partial Pivoting:** For each column, find the row at or below `pivot_row` with the largest absolute value in that column and swap it to the current pivot row. This ensures numerical stability.
   * **Zero-Pivot Detection:** If the largest candidate pivot is below our threshold ($10^{-12}$), the column corresponds to a free variable. We skip row reduction for this column and proceed to the next column without incrementing `pivot_row`.
   * **Elimination:** Subtract appropriate scalar multiples of the pivot row from all rows below it to create zeros under the pivot: $\text{factor} = \text{aug}[r][\text{col}] / \text{pivot\_val}$.

2. **Backward Elimination (Reduced Row Echelon Form - RREF):**
   * **Pivots to 1:** Scale each pivot row by dividing by its leading entry.
   * **Elimination above & below:** Eliminate all non-zero entries in the pivot column both above and below the pivot, leaving exactly $1.0$ at the pivot position and $0.0$ elsewhere in that column.


In [40]:
def compute_ref_pure(A, b):
    """
    Reduces the augmented matrix [A|b] to Row Echelon Form (REF) using pure Python.
    Tracks pivot positions as (row, col) to correctly handle non-sequential cases.
    """
    m = len(A)
    n = len(A[0])
    
    # Construct augmented matrix [A|b]
    aug = []
    for i in range(m):
        aug.append(A[i] + [b[i]])
        
    pivot_row = 0
    pivot_positions = []
    
    # Forward elimination column-by-column
    for col in range(n):
        if pivot_row >= m:
            break
            
        # Find pivot candidate (partial pivoting)
        best_row = pivot_row
        best_val = abs(aug[pivot_row][col])
        for r in range(pivot_row + 1, m):
            if abs(aug[r][col]) > best_val:
                best_val = abs(aug[r][col])
                best_row = r
                
        # Zero-pivot check: if best pivot is near zero, it is a free column
        if best_val < 1e-12:
            continue
            
        # Swap current row with best_row
        if best_row != pivot_row:
            aug[pivot_row], aug[best_row] = aug[best_row], aug[pivot_row]
            
        pivot_positions.append((pivot_row, col))
        
        # Eliminate entries below the pivot
        for r in range(pivot_row + 1, m):
            if abs(aug[r][col]) > 1e-12:
                pivot_entry = aug[pivot_row][col]
                factor = aug[r][col] / pivot_entry
                for j in range(col, n + 1):
                    aug[r][j] -= factor * aug[pivot_row][j]
                    
        pivot_row += 1
        
    return aug, pivot_positions

def compute_rref_pure(ref_matrix, pivot_positions):
    """
    Reduces an augmented matrix in REF to Reduced Row Echelon Form (RREF) using pure Python.
    """
    m = len(ref_matrix)
    n = len(ref_matrix[0]) - 1 # excluding augmented column
    
    rref = [[val for val in row] for row in ref_matrix]
    
    # Backward elimination from right to left
    for p_row, p_col in reversed(pivot_positions):
        pivot_val = rref[p_row][p_col]
        
        if abs(pivot_val) < 1e-12:
            continue
            
        # Scale pivot row to leading 1
        for j in range(p_col, n + 1):
            rref[p_row][j] /= pivot_val
            
        # Eliminate entries in pivot column
        for r in range(m):
            if r != p_row and abs(rref[r][p_col]) > 1e-12:
                factor = rref[r][p_col]
                for j in range(p_col, n + 1):
                    rref[r][j] -= factor * rref[p_row][j]
                    
    return rref


### 1.2 Identifying Pivots, Particular Solutions, and the Null Space

**Mathematical Formulations:**

* **Pivot Columns:** Columns with a leading $1.0$ in RREF.
* **Non-Pivot (Free) Columns:** Columns without a leading pivot. These represent free variables in the system.
* **Particular Solution ($x_p$):**
  * Set all free variables to $0.0$.
  * Read pivot variables from the final column of the RREF augmented matrix: $x_{p}[p_{\text{col}}] = \text{RREF}[p_{\text{row}}][-1]$.
* **Null-Space Basis (Solutions to $Ax = 0$):**
  * For each free variable column index $j$, construct a basis vector $v$ by setting $x_j = 1.0$ and all other free variables to $0.0$.
  * Back-solve for the pivot variables: $x_{p_{\text{col}}} = -\text{RREF}[p_{\text{row}}][j]$.
  * This yields a set of linearly independent vectors spanning the null space of $A$.
* **General Solution ($x_g$):**
  * Combines the particular solution and the null-space span:
    $$x_g = x_p + \sum_{i=1}^{k} c_i v_i$$
    where $c_i$ are arbitrary coefficients and $v_i$ are the basis vectors of the null space.


In [41]:
def pivot_and_free_columns_pure(n, pivot_positions):
    """Splits the column indices of A into pivot columns and free (non-pivot) columns."""
    pivot_cols = [col for (_, col) in pivot_positions]
    free_cols = [c for c in range(n) if c not in pivot_cols]
    return pivot_cols, free_cols

def particular_solution_pure(rref_matrix, pivot_positions, free_cols, n):
    """Computes a particular solution Ax = b with all free variables set to 0.0."""
    x_p = [0.0 for _ in range(n)]
    for p_row, p_col in pivot_positions:
        # Since free variables are set to 0, the pivot variables equal the corresponding
        # entry in the last column (the b-column) of the RREF matrix.
        x_p[p_col] = rref_matrix[p_row][-1]
    return x_p

def nullspace_basis_pure(rref_matrix, pivot_positions, free_cols, n):
    """
    Computes a basis for the null space (solutions to Ax = 0) by setting each free
    variable to 1.0 (and others to 0.0) and back-solving the pivot variables from RREF.
    """
    basis = []
    for free_col in free_cols:
        v = [0.0 for _ in range(n)]
        v[free_col] = 1.0
        for p_row, p_col in pivot_positions:
            # Pivot row equation: 1.0 * x_{p_col} + sum_{j in free} rref[p_row][j] * x_j = 0
            # Since only free_col is 1.0 and other free variables are 0.0, this simplifies to:
            # x_{p_col} = -rref[p_row][free_col]
            v[p_col] = -rref_matrix[p_row][free_col]
        basis.append(v)
    return basis

def general_solution_pure(x_p, null_basis, coeffs):
    """Constructs the general solution as x_p + c1*v1 + c2*v2 + ..."""
    x = [val for val in x_p]
    for c, v in zip(coeffs, null_basis):
        for j in range(len(x)):
            x[j] += c * v[j]
    return x


### 1.3 Worked Numerical Example (5 × 7 System)

This section demonstrates the system using a specific $5 \times 7$ matrix $A$ and a $5 \times 1$ vector $b$, with values formatted with exactly 8 decimal places.

Since standard utility imports (like `math`) are avoided here to ensure self-containment, we implement a Babylonian (Newton-Raphson) square root function to compute $L_2$ vector norms.

We will compute and display:
1. Augmented matrix $[A \mid b]$ reduced to REF.
2. Reduced Row Echelon Form (RREF).
3. Pivot and free columns.
4. Particular solution $x_p$.
5. Null-space basis vectors $v_1$ and $v_2$.
6. General solution $x_g$ with chosen coefficients.
7. Verification of solutions by computing the residual norms: $\|A x_p - b\|_2$, $\|A v_i\|_2$, and $\|A x_g - b\|_2$.


In [42]:
# 5x7 coefficient matrix A and 5x1 vector b with random decimal values
A = [
    [ 4.93181539, -1.10025731,  6.45501847,  3.55394208, -7.30519451,  8.56193751,  4.70104812],
    [ 5.14923485, -6.69481943, -0.89319485, -2.32681539,  7.68214510,  2.59083751,  5.81094851],
    [-1.01948512, -4.91024513,  0.98381945, -7.85104812,  5.89793145,  2.37014581,  4.64681539],
    [-2.61983751,  8.47394812,  7.07614581,  5.01104812, -5.49781539, -0.59925731, -8.21219451],
    [-6.22319485,  3.29583751,  4.40694812,  8.41514581, -3.13581539, -2.33225731, -0.54804812]
]

b = [2.25281539, 7.15024513, 4.96204812, -4.94681539, -3.59793751]

def manual_sqrt(x):
    """
    Computes square root of x using the Babylonian (Newton-Raphson) method.
    """
    if x < 0.0:
        raise ValueError("Cannot compute square root of negative numbers")
    if x == 0.0:
        return 0.0
    guess = x / 2.0
    for _ in range(100):
        next_guess = 0.5 * (guess + x / guess)
        diff = abs(next_guess - guess)
        if diff < 1e-15:
            return next_guess
        guess = next_guess
    return guess

def print_matrix(M, name="Matrix"):
    print(f"{name}:")
    for row in M:
        print("  [" + ", ".join(f"{x:12.8f}" for x in row) + "]")
    print()

def print_vector(v, name="Vector"):
    print(f"{name}:")
    print("  [" + ", ".join(f"{x:12.8f}" for x in v) + "]")
    print()

def mat_vec_multiply(M, v):
    """Matrix-vector multiplication using nested loops."""
    m = len(M)
    n = len(M[0])
    res = [0.0 for _ in range(m)]
    for i in range(m):
        for j in range(n):
            res[i] += M[i][j] * v[j]
    return res

def vec_subtract(v1, v2):
    """Vector subtraction."""
    return [x1 - x2 for x1, x2 in zip(v1, v2)]

def vec_norm_l2(v):
    """Vector L2 norm using the custom Babylonian manual_sqrt."""
    return manual_sqrt(sum(x*x for x in v))

print_matrix(A, "Generated Matrix A (5x7) - Shown with full precision")
print_vector(b, "Generated Vector b (5x1) - Shown with full precision")

# Compute REF
ref_matrix, pivot_pos = compute_ref_pure(A, b)
print_matrix(ref_matrix, "REF of Augmented Matrix [A | b]")
print("Pivot Positions discovered (row, column):", pivot_pos, "\n")

# Compute RREF
rref_matrix = compute_rref_pure(ref_matrix, pivot_pos)
print_matrix(rref_matrix, "RREF of Augmented Matrix [A | b]")

# Pivot and free columns
pivot_cols, free_cols = pivot_and_free_columns_pure(7, pivot_pos)
print("Pivot Columns:", pivot_cols)
print("Non-Pivot (Free) Columns:", free_cols, "\n")

# Particular solution
x_p = particular_solution_pure(rref_matrix, pivot_pos, free_cols, 7)
print_vector(x_p, "Particular Solution x_p")

# Null space basis
basis = nullspace_basis_pure(rref_matrix, pivot_pos, free_cols, 7)
for idx, v in enumerate(basis):
    print_vector(v, f"Nullspace Basis Vector v_{idx+1}")

# Chosen coefficients for the general solution
coeffs = [-3.71428519, -0.00714581]
print("Arbitrary Random Coefficients chosen for the nullspace vectors:")
for idx, c in enumerate(coeffs):
    print(f"  c_{idx+1} = {c:11.8f}")
print()

x_g = general_solution_pure(x_p, basis, coeffs)
print_vector(x_g, "General Solution x_g = x_p + c1*v1 + c2*v2")

# Verification of solutions (residuals)
print("=================== MATHEMATICAL VERIFICATION ===================")
# Residual of the particular solution
Ax_p = mat_vec_multiply(A, x_p)
diff_p = vec_subtract(Ax_p, b)
error_p = vec_norm_l2(diff_p)
print(f"Residual error of Particular Solution ||Ax_p - b||_2: {error_p:.8e}")
print_vector(Ax_p, "Ax_p (should match b exactly)")

# Residuals of null-space basis vectors
for idx, v in enumerate(basis):
    Av = mat_vec_multiply(A, v)
    error_v = vec_norm_l2(Av)
    print(f"Residual error of homogeneous solution ||Av_{idx+1}||_2: {error_v:.8e}")

# Residual of the general solution
Ax_g = mat_vec_multiply(A, x_g)
diff_g = vec_subtract(Ax_g, b)
error_g = vec_norm_l2(diff_g)
print(f"\nResidual error of General Solution ||Ax_g - b||_2: {error_g:.8e}")
print_vector(Ax_g, "Ax_g (should match b exactly)")


Generated Matrix A (5x7) - Shown with full precision:
  [  4.93181539,  -1.10025731,   6.45501847,   3.55394208,  -7.30519451,   8.56193751,   4.70104812]
  [  5.14923485,  -6.69481943,  -0.89319485,  -2.32681539,   7.68214510,   2.59083751,   5.81094851]
  [ -1.01948512,  -4.91024513,   0.98381945,  -7.85104812,   5.89793145,   2.37014581,   4.64681539]
  [ -2.61983751,   8.47394812,   7.07614581,   5.01104812,  -5.49781539,  -0.59925731,  -8.21219451]
  [ -6.22319485,   3.29583751,   4.40694812,   8.41514581,  -3.13581539,  -2.33225731,  -0.54804812]

Generated Vector b (5x1) - Shown with full precision:
  [  2.25281539,   7.15024513,   4.96204812,  -4.94681539,  -3.59793751]

REF of Augmented Matrix [A | b]:
  [ -6.22319485,   3.29583751,   4.40694812,   8.41514581,  -3.13581539,  -2.33225731,  -0.54804812,  -3.59793751]
  [  0.00000000,   7.08646806,   5.22091096,   1.46844417,  -4.17770137,   0.38257523,  -7.98147748,  -3.43215742]
  [  0.00000000,   0.00000000,   8.83377268,   9.

## Q2) Dataset, Rank, and the Power Method

### 2.1 Dataset Generation

We construct a dataset $X \in \mathbb{R}^{500 \times 6}$ where the first four features $f_1, f_2, f_3, f_4$ are standard normal, $f_i \sim \mathcal{N}(0, 1)$, generated with 500 samples each.
The fifth and sixth features are defined by the linear relationships:
$$f_5 = 2f_1 + 3f_2$$
$$f_6 = f_3 - 2f_4$$

We use NumPy to generate the standard normal variables and apply the relations to construct the final dataset $X$.


In [ ]:
!pip install numpy pandas matplotlib seaborn scikit-learn

In [44]:
import numpy as np

def generate_dataset(n_samples=500, seed=2026):
    """
    Generates dataset X in R^(500x6) where the first four features are standard normal,
    and the fifth and sixth features are linear combinations of the first four.
    """
    rng = np.random.default_rng(seed)
    f1 = rng.standard_normal(n_samples)
    f2 = rng.standard_normal(n_samples)
    f3 = rng.standard_normal(n_samples)
    f4 = rng.standard_normal(n_samples)
    
    f5 = 2.0 * f1 + 3.0 * f2
    f6 = f3 - 2.0 * f4
    
    return np.column_stack([f1, f2, f3, f4, f5, f6])

X = generate_dataset(500, seed=2026)
print("Dataset X shape:", X.shape)
print("\nFirst 5 rows of the generated dataset X:")
for i in range(5):
    print("  [" + ", ".join(f"{val:12.8f}" for val in X[i]) + "]")


Dataset X shape: (500, 6)

First 5 rows of the generated dataset X:
  [ -0.79312248,   2.05811529,   0.39745116,  -1.23827117,   4.58810093,   2.87399350]
  [  0.24057128,   0.70612604,   0.83637692,   0.32769420,   2.59952069,   0.18098853]
  [ -1.89632635,  -1.09601177,   0.41566281,   0.67665505,  -7.08068800,  -0.93764730]
  [  1.39577171,  -1.13127226,   0.69040168,   0.69287634,  -0.60227337,  -0.69535099]
  [  0.63829474,  -0.95273569,  -1.17145823,  -0.32029866,  -1.58161758,  -0.53086091]


### 2.2 Rank of $X$

We compute the rank of $X$ using `np.linalg.matrix_rank`. 

Since the columns $f_5$ and $f_6$ are exact linear combinations of $f_1, f_2, f_3, f_4$, the column space is spanned by the first four linearly independent features. Hence, the theoretical rank is exactly **4**.


In [45]:
rank = np.linalg.matrix_rank(X)
print(f"Computed Rank of X: {rank}")
print("This matches the theoretical rank of 4, as f5 and f6 are linearly dependent on the first four features.")


Computed Rank of X: 4
This matches the theoretical rank of 4, as f5 and f6 are linearly dependent on the first four features.


### 2.3 Numerical Experiment with the Power Method

#### (a) Covariance Matrix $C = \frac{1}{n} X^T X$

We compute the covariance matrix $C$ of the dataset $X$ using loops and basic arithmetic, avoiding library matrix multiplication functions.


In [46]:
def compute_covariance_pure(X):
    """
    Computes the biased covariance matrix C = (1/n) * X^T * X manually.
    """
    n = len(X)
    p = len(X[0])
    
    X_list = [[float(val) for val in row] for row in X]
    C = [[0.0 for _ in range(p)] for _ in range(p)]
    
    # Outer product summation
    for i in range(p):
        for j in range(p):
            sum_val = 0.0
            for k in range(n):
                sum_val += X_list[k][i] * X_list[k][j]
            C[i][j] = sum_val / n
            
    return C

C = compute_covariance_pure(X)

print("Covariance Matrix C (6x6):")
for row in C:
    print("  [" + ", ".join(f"{x:12.8f}" for x in row) + "]")


Covariance Matrix C (6x6):
  [  1.09686422,   0.00223918,   0.01014359,   0.02663460,   2.20044596,  -0.04312560]
  [  0.00223918,   1.02770095,   0.04796297,   0.05554491,   3.08758120,  -0.06312685]
  [  0.01014359,   0.04796297,   0.93584133,  -0.06721832,   0.16417611,   1.07027797]
  [  0.02663460,   0.05554491,  -0.06721832,   0.98713229,   0.21990393,  -2.04148290]
  [  2.20044596,   3.08758120,   0.16417611,   0.21990393,  13.66363553,  -0.27563176]
  [ -0.04312560,  -0.06312685,   1.07027797,  -2.04148290,  -0.27563176,   5.15324376]


#### (b) Power Method for Largest Eigenvalue $\lambda_1$ and Eigenvector $v_1$

We implement the Power Method to approximate the dominant eigenvalue $\lambda_1$ and its corresponding eigenvector $v_1$ of $C$.

**Algorithm:**
1. **Initialization:** Start with a random unit vector $v \in \mathbb{R}^6$.
2. **Matrix Multiplication:** Compute $w = C v$ at each iteration.
3. **Normalization:** Check if $\|w\|_2$ is near zero to prevent division by zero. Otherwise, update $v \leftarrow w / \|w\|_2$.
4. **Rayleigh Quotient:** Estimate the eigenvalue: $\lambda = v^T C v$.
5. **Convergence Check:** Terminate when $|\lambda - \lambda_{\text{old}}| < 10^{-7}$.


In [47]:
import numpy as np
import math

def power_method_pure(C, tol=1e-7, max_iter=10000, seed=1):
    """
    Estimates the dominant eigenvalue and eigenvector of symmetric matrix C by power iteration.
    """
    p = len(C)
    
    # Start with a random normal vector
    rng = np.random.default_rng(seed)
    v_np = rng.standard_normal(p)
    v = [float(x) for x in v_np]
    
    norm_v = math.sqrt(sum(x*x for x in v))
    if norm_v > 1e-12:
        v = [x / norm_v for x in v]
    else:
        v = [1.0 if idx == 0 else 0.0 for idx in range(p)]
        
    lam_old = 0.0
    
    for iters in range(1, max_iter + 1):
        # Matrix-vector multiplication w = C * v
        w = [0.0 for _ in range(p)]
        for i in range(p):
            for j in range(p):
                w[i] += C[i][j] * v[j]
                
        # Normalize
        norm_w = math.sqrt(sum(x*x for x in w))
        if norm_w < 1e-12:
            lam = 0.0
            break
            
        v = [x / norm_w for x in w]
        
        # Rayleigh quotient lam = v^T * C * v
        Cv = [0.0 for _ in range(p)]
        for i in range(p):
            for j in range(p):
                Cv[i] += C[i][j] * v[j]
                
        lam = sum(v[i] * Cv[i] for i in range(p))
        
        if abs(lam - lam_old) < tol:
            break
        lam_old = lam
        
    return lam, v, iters

lam1, v1, iters1 = power_method_pure(C, tol=1e-7, max_iter=10000, seed=1)

print(f"Dominant Eigenvalue λ_1: {lam1:.8f}")
print("Corresponding Eigenvector v_1:")
print("  [" + ", ".join(f"{x:12.8f}" for x in v1) + "]")
print(f"Iterations to converge: {iters1}")


Dominant Eigenvalue λ_1: 14.73187872
Corresponding Eigenvector v_1:
  [  0.15556618,   0.21721700,   0.00964094,   0.02147799,   0.96278336,  -0.03331504]
Iterations to converge: 12


#### (c) Deflation Method for Remaining Eigenvalues and Eigenvectors

To find the subsequent eigenvalues and eigenvectors, we deflate the covariance matrix to subtract the components of the already computed eigenvectors:
$$C_{\text{work}} = C - \sum_{j=1}^{k-1} v_j v_j^T C$$

In each deflation step, the outer product update $v_j (v_j^T C_{\text{work}})$ is computed manually:
1. Compute the row vector $u^T = v_j^T C_{\text{work}}$ where $u_j = \sum_{i=1}^{p} v_{j, i} C_{\text{work}}[i][j]$.
2. Subtract $v_{j, i} u_j$ from $C_{\text{work}}[i][j]$.

This allows us to successively find the remaining eigenvalues and eigenvectors by running the Power Method on the updated $C_{\text{work}}$.


In [48]:
def power_method_all_pure(C, tol=1e-7, max_iter=10000):
    """
    Recovers all eigenpairs of C by sequential matrix deflation.
    """
    p = len(C)
    C_work = [[val for val in row] for row in C]
    
    eigvals = []
    eigvecs = []
    iters_list = []
    
    for k in range(p):
        lam, v, iters = power_method_pure(C_work, tol=tol, max_iter=max_iter, seed=100 + k)
        eigvals.append(lam)
        eigvecs.append(v)
        iters_list.append(iters)
        
        # Deflation: C_work = C_work - v * v^T * C_work
        # Compute row vector u^T = v^T * C_work
        u = [0.0 for _ in range(p)]
        for j in range(p):
            for i in range(p):
                u[j] += v[i] * C_work[i][j]
                
        # Outer product subtraction
        for i in range(p):
            for j in range(p):
                C_work[i][j] -= v[i] * u[j]
                
    return eigvals, eigvecs, iters_list

eigvals, eigvecs, iters_list = power_method_all_pure(C, tol=1e-7, max_iter=10000)

print("Eigenvalues from Power Method (with deflation):")
for idx, (lam, iters) in enumerate(zip(eigvals, iters_list)):
    print(f"  λ_{idx+1} = {lam:12.8f}  (converged in {iters:4d} iterations)")

print("\nCorresponding Eigenvectors (as columns):")
print("  " + "  ".join(f"     v_{i+1}   " for i in range(6)))
for r in range(6):
    row_str = "  [" + ", ".join(f"{eigvecs[col][r]:12.8f}" for col in range(6)) + "]"
    print(row_str)


Eigenvalues from Power Method (with deflation):
  λ_1 =  14.73187872  (converged in   12 iterations)
  λ_2 =   6.17303361  (converged in    7 iterations)
  λ_3 =   1.07508567  (converged in   31 iterations)
  λ_4 =   0.88442008  (converged in    2 iterations)
  λ_5 =   0.00000000  (converged in    1 iterations)
  λ_6 =   0.00000000  (converged in    1 iterations)

Corresponding Eigenvectors (as columns):
       v_1          v_2          v_3          v_4          v_5          v_6   
  [ -0.15556587,   0.00593547,  -0.82510095,   0.09621939,   0.55640208,  -0.02450590]
  [ -0.21721660,   0.00771784,   0.55221410,  -0.07042607,   0.55450542,   0.48244392]
  [ -0.00963100,   0.19222656,   0.10651637,   0.88597054,  -0.61612872,   0.76776646]
  [ -0.02149659,  -0.35999317,   0.05366690,   0.44765613,  -0.00971526,  -0.02061332]
  [ -0.96278155,   0.03502446,   0.00644039,  -0.01883944,   0.00053246,  -0.09236222]
  [  0.03336218,   0.91221289,  -0.00081743,  -0.00934173,   0.05684362,   0.4

#### (d) Comparison with Library-Based Results

To verify the accuracy of our manual implementation, we compare our results against `numpy.linalg.eigh`, which is the standard solver for symmetric matrices.

We will compute:
1. Eigenvalues using NumPy.
2. Absolute differences between our manual eigenvalues and the library eigenvalues.
3. Absolute dot products between the corresponding eigenvectors, which should be close to $1.0$ if the vectors are aligned.


In [49]:
import numpy as np

C_np = np.array(C)
# C is symmetric, so eigh is appropriate
eigvals_np, eigvecs_np = np.linalg.eigh(C_np)

# Sort eigenvalues and eigenvectors in descending order of magnitude
order = np.argsort(eigvals_np)[::-1]
eigvals_np = eigvals_np[order]
eigvecs_np = eigvecs_np[:, order]

print("Eigenvalues from numpy.linalg.eigh:")
for idx, lam in enumerate(eigvals_np):
    print(f"  λ_{idx+1} (numpy) = {lam:12.8f}")
print()

print("Comparison of Eigenvalues (Absolute Differences):")
for idx in range(6):
    diff = abs(eigvals[idx] - eigvals_np[idx])
    print(f"  |λ_{idx+1}(power) - λ_{idx+1}(numpy)|: {diff:.8e}")
print()

print("Comparison of Eigenvectors (Absolute Dot Products):")
for idx in range(6):
    v_pow = eigvecs[idx]
    v_np = eigvecs_np[:, idx]
    dot = abs(sum(v_pow[j] * v_np[j] for j in range(6)))
    print(f"  |v_{idx+1}(power) . v_{idx+1}(numpy)|: {dot:12.8f}")


Eigenvalues from numpy.linalg.eigh:
  λ_1 (numpy) =  14.73187872
  λ_2 (numpy) =   6.17303360
  λ_3 (numpy) =   1.07508583
  λ_4 (numpy) =   0.88441993
  λ_5 (numpy) =   0.00000000
  λ_6 (numpy) =   0.00000000

Comparison of Eigenvalues (Absolute Differences):
  |λ_1(power) - λ_1(numpy)|: 5.22273780e-09
  |λ_2(power) - λ_2(numpy)|: 4.19633484e-09
  |λ_3(power) - λ_3(numpy)|: 1.53636293e-07
  |λ_4(power) - λ_4(numpy)|: 1.54662699e-07
  |λ_5(power) - λ_5(numpy)|: 1.34220486e-15
  |λ_6(power) - λ_6(numpy)|: 3.50885250e-16

Comparison of Eigenvectors (Absolute Dot Products):
  |v_1(power) . v_1(numpy)|:   1.00000000
  |v_2(power) . v_2(numpy)|:   1.00000000
  |v_3(power) . v_3(numpy)|:   0.99999960
  |v_4(power) . v_4(numpy)|:   0.99999960
  |v_5(power) . v_5(numpy)|:   0.71141981
  |v_6(power) . v_6(numpy)|:   0.12211880


#### (e) Iterations Required and Convergence Analysis

Our power iteration implementation uses a tolerance of $10^{-7}$ for successive Rayleigh quotients. Below are the iteration counts for each eigenvalue:

* **$\lambda_1$:** 12 iterations
* **$\lambda_2$:** 7 iterations
* **$\lambda_3$:** 31 iterations
* **$\lambda_4$:** 2 iterations
* **$\lambda_5$:** 1 iteration
* **$\lambda_6$:** 1 iteration

### Analysis of Convergence Behavior

1. **Spectral Gap and Rate of Convergence:** 
   The convergence rate of the Power Method is linear and is governed by the ratio of the two largest eigenvalues of the working matrix: $\left| \frac{\lambda_{k+1}}{\lambda_k} \right|$.
   * A small ratio (large spectral gap) leads to fast convergence.
   * A ratio close to 1 (narrow spectral gap) results in slower convergence.

2. **Spectral Gap Analysis:**
   * **$\lambda_1 \approx 14.73$ and $\lambda_2 \approx 6.17$:** The ratio $\left| \frac{\lambda_2}{\lambda_1} \right| \approx 0.42$ is small, reflecting a large spectral gap. $\lambda_1$ converges rapidly in 12 iterations.
   * **$\lambda_3 \approx 1.075$ and $\lambda_4 \approx 0.884$:** The ratio $\left| \frac{\lambda_4}{\lambda_3} \right| \approx 0.82$ is close to 1, representing a narrow gap. This explains why $\lambda_3$ requires significantly more iterations (31) to converge.
   * **$\lambda_5 \approx 0$ and $\lambda_6 \approx 0$:** Since the dataset $X$ has rank 4, the true covariance matrix has a 2-dimensional null space, meaning the last two eigenvalues are exactly $0$. The initial random vector yields a Rayleigh quotient of the order of $10^{-15}$ on the first iteration. The convergence criteria of $10^{-7}$ is therefore met on the very first iteration, reporting convergence in **1 iteration**.

3. **Orthonormal Bases in Eigenspaces:**
   * For the degenerate null space corresponding to $\lambda \approx 0$, any orthonormal basis within this 2D space constitutes a valid set of eigenvectors.
   * Because the eigenspace is degenerate, our sequential deflation path and NumPy's eigensolver can produce different orthonormal vectors within this subspace.
   * This explains why the absolute dot products for $v_5$ and $v_6$ with the library eigenvectors are not equal to $1.0$. This is mathematically correct and expected.
